# Task 3: Caching Experiment (Queries 1, 6)

- Caches a reduced `integrated_taxi_trips` view containing only the columns used by Queries 1 and 6.
- Compares Query 1 and Query 6 before and after caching.
- Evaluates a shared monthly aggregation cache to eliminate redundant computations.

## Connect to Spark and imports

In [ ]:
import sys
from time import perf_counter
from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import (
    INTEGRATED_TABLE_VIEW,
    execute_registered_query,
    register_integrated_taxi_trips,
)

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-Caching-Queries1-6')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')

QUERY_NAMES = (
    'query_01_monthly_zone_demand',
    'query_06_monthly_taxi_demand_trend',
)
REPETITIONS = 9

## Measure uncached execution

In [13]:
spark.catalog.clearCache()
register_integrated_taxi_trips(spark)

uncached_results = {
    query_name: measure_registered_query(spark, query_name, REPETITIONS)
    for query_name in QUERY_NAMES
}

print('Uncached measurements complete.')

Uncached measurements complete.


In [14]:
for query_name in QUERY_NAMES:
    print(f'Uncached plan: {query_name}')
    execute_registered_query(spark, query_name).explain('formatted')

Uncached plan: query_01_monthly_zone_demand
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pickup_date#3205, pickup_zone_name#3212, date#3246]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
ReadSchema: struct<pickup_date:string,pickup_zone_name:string>

(2) Project
Output [2]: [pickup_zone_name#3212, substring(pickup_date#3205, 1, 7) AS pickup_month#4428]
Input [3]: [pickup_date#3205, pickup_zone_name#3212, date#3246]

(3) HashAggregate
Input [2]: [pickup_zone_name#3212, pickup_month#4428]
Keys [2]: [pickup_zone_name#3212, pickup_month#4428]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#4559L]
Results [3]: [pickup_zone_name#3212, pickup_month#4428, count#4560L]

(4) Exchange
Input [3]: [pickup_zone_name#32

## Cache a reduced shared input

- Caching the full Gold table exceeds available executor memory.
- Queries 1 and 6 need only the below two columns.

In [15]:
CACHE_COLUMNS = ('pickup_date', 'pickup_zone_name')

cache_input = spark.table(INTEGRATED_TABLE_VIEW).select(*CACHE_COLUMNS)
cache_input.createOrReplaceTempView(INTEGRATED_TABLE_VIEW)

cache_started_at = perf_counter()
spark.catalog.cacheTable(INTEGRATED_TABLE_VIEW)
cached_input_rows = spark.table(INTEGRATED_TABLE_VIEW).count()
cache_materialization_seconds = perf_counter() - cache_started_at

assert spark.catalog.isCached(INTEGRATED_TABLE_VIEW)
print(f'Cached input rows: {cached_input_rows}')
print(f'Cache materialization: {cache_materialization_seconds:.3f} seconds')

Cached input rows: 9417833
Cache materialization: 5.512 seconds


## Measure cached execution and verify results

In [16]:
cached_results = {
    query_name: measure_registered_query(spark, query_name, REPETITIONS)
    for query_name in QUERY_NAMES
}

for query_name in QUERY_NAMES:
    assert rows_match(
        uncached_results[query_name]['rows'],
        cached_results[query_name]['rows'],
    ), (
        f'Results differ for {query_name}.'
    )

print('All cached results match the uncached results within float tolerance.')

All cached results match the uncached results within float tolerance.


In [17]:
for query_name in QUERY_NAMES:
    print(f'Cached plan: {query_name}')
    execute_registered_query(spark, query_name).explain('formatted')

Cached plan: query_01_monthly_zone_demand
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Exchange (8)
            +- HashAggregate (7)
               +- Project (6)
                  +- Scan In-memory table integrated_taxi_trips (1)
                        +- InMemoryRelation (2)
                              +- * Project (5)
                                 +- * ColumnarToRow (4)
                                    +- Scan parquet  (3)


(1) Scan In-memory table integrated_taxi_trips
Output [2]: [pickup_date#3205, pickup_zone_name#3212]
Arguments: [pickup_date#3205, pickup_zone_name#3212]

(2) InMemoryRelation
Arguments: [pickup_date#3205, pickup_zone_name#3212], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan parquet 
Output [3]: [pickup_date#3205, pickup_zone_name#3212, date#3246]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
ReadSchema: struct<picku

## Advanced Caching: Shared Monthly Aggregation

- In this section, we test a more aggressive caching strategy.
- Instead of just caching the raw columns, we cache the result of the aggregation by month and zone.
- This result is then used as a shared intermediate step for both Query 1 and Query 6, eliminating redundant computation across both queries.

In [ ]:
print("\nMeasuring Shared Monthly Aggregation Caching...")
spark.catalog.clearCache()
register_integrated_taxi_trips(spark)

# 1. Create the intermediate shared aggregation
shared_agg_view = "shared_monthly_zone_agg"
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW {shared_agg_view} AS
    SELECT
        substring(pickup_date, 1, 7) as pickup_month,
        pickup_zone_name,
        count(*) as demand
    FROM integrated_taxi_trips
    GROUP BY 1, 2
""")

# 2. Cache and Materialize
start_mat = perf_counter()
spark.catalog.cacheTable(shared_agg_view)
spark.table(shared_agg_view).count()
mat_time = perf_counter() - start_mat
print(f"Shared Agg Materialization Time: {mat_time:.3f}s")

# 3. Measure Shared Cache Performance using repetitions for consistency
shared_results = {}
for name in QUERY_NAMES:
    samples = []
    for _ in range(REPETITIONS):
        if name == 'query_01_monthly_zone_demand':
            start = perf_counter()
            spark.table(shared_agg_view).orderBy("pickup_zone_name", "pickup_month").collect()
        else: # query_06
            start = perf_counter()
            spark.table(shared_agg_view).groupBy("pickup_month").sum("demand").orderBy("pickup_month").collect()
        samples.append(perf_counter() - start)

    import statistics
    shared_results[name] = {
        'median_seconds': statistics.median(samples),
        'samples_seconds': samples
    }

print("Shared Aggregation measurements complete.")


Measuring Shared Monthly Aggregation Caching...


Shared Agg Materialization Time: 3.176s
Shared Aggregation measurements complete.


## Final Benchmark Comparison

Comparing Baseline vs. Column-based Cache vs. Shared Aggregation (Co-cache).

In [19]:
print(f"{'Query':<35} | {'Baseline':<12} | {'Col-Cache':<12} | {'Co-Cache':<12} | {'Improvement (Col vs Co)':<20}")
print("-" * 95)

for q in QUERY_NAMES:
    base = uncached_results[q]['median_seconds']
    col = cached_results[q]['median_seconds']
    co = shared_results[q]['median_seconds']
    diff = col - co

    print(f"{q:<35} | {base:>11.3f}s | {col:>11.3f}s | {co:>11.3f}s | {diff:>18.3f}s")

print("\nNote: 'Improvement' shows the time saved by using Shared Aggregation over Column-based caching.")

Query                               | Baseline     | Col-Cache    | Co-Cache     | Improvement (Col vs Co)
-----------------------------------------------------------------------------------------------
query_01_monthly_zone_demand        |       3.974s |       1.157s |       0.175s |              0.982s
query_06_monthly_taxi_demand_trend  |       2.481s |       0.795s |       0.327s |              0.469s

Note: 'Improvement' shows the time saved by using Shared Aggregation over Column-based caching.


In [20]:
spark.catalog.uncacheTable(INTEGRATED_TABLE_VIEW)
spark.catalog.clearCache()
spark.stop()